# Anomaly Detection in Financial Transactions

**End Course Summative Assignment - Advanced Machine Learning**  
**Domain:** Banking and Finance  
**Author:** Bijitesh Kanrar  

## Executive summary

This project develops an operationally realistic fraud-screening pipeline for highly
imbalanced credit-card transactions. It compares a transparent statistical baseline,
Isolation Forest, Local Outlier Factor, and PCA reconstruction error. A
class-imbalance-aware Random Forest is included as a supervised reference where labels
are available.

Model selection is based on validation data only. The decision threshold explicitly
trades the cost of missed fraud against the cost of manual investigation. The locked
threshold is then evaluated once on a time-ordered test set using PR-AUC, ROC-AUC,
precision, recall, F1, F2, false-positive rate, and confusion-matrix counts.

## Assessment-criteria mapping

| Requirement | Evidence in this notebook |
|---|---|
| Preprocessing and cleaning | Schema validation, missing-value audit, duplicate removal, robust transformation |
| Handling imbalance | PR-AUC/F2 metrics, cost-sensitive threshold, normal-only detector training, SMOTE or class-weighted benchmark |
| Multiple outlier techniques | IQR, Isolation Forest, Local Outlier Factor, PCA reconstruction error |
| Model comparison | Validation-only tuning and one-time locked test evaluation |
| Advanced analysis | Time-ordered split, leakage control, top-k review capacity, threshold economics |
| Practical deployment | Drive-saved model bundle, notebook scoring example, monitoring and governance plan |

### Research questions

1. Which anomaly detector retrieves the most fraud while keeping investigation volume manageable?
2. How much value does labelled supervised learning add over novelty detection?
3. How should the alert threshold change when missed fraud costs more than a false alert?

## Dataset and attribution

The final analysis uses the anonymized **ULB/Worldline Credit Card Fraud Detection**
dataset. It contains European card transactions recorded over two days, including 492
labelled frauds among 284,807 transactions. `V1`-`V28` are PCA-anonymized attributes;
`Time`, `Amount`, and `Class` are also provided.

Source: [Kaggle - Machine Learning Group, ULB](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

The project does not claim that this historical benchmark represents current banking
traffic. It is used to demonstrate sound anomaly-detection methodology. The notebook
may generate synthetic data only when explicitly enabled for a smoke test; synthetic
metrics must never be submitted as real results.

In [ ]:
# Run this setup cell in a fresh Google Colab runtime.
import subprocess
import sys
from pathlib import Path
from google.colab import drive

REPO_URL = "https://github.com/CA-BijiteshKanrar/ADFT-BK.git"
REPO_DIR = Path("/content/ADFT-BK")
if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git clone.")
    print(f"Reusing GitHub clone: {REPO_DIR}")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True,
    )
    print(f"Cloned GitHub project to: {REPO_DIR}")

# All analysis functions are embedded below; no code is imported from the clone.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "numpy>=1.26,<3", "pandas>=2.1,<3", "scikit-learn>=1.4,<2",
        "imbalanced-learn>=0.12,<1", "matplotlib>=3.8,<4",
        "seaborn>=0.13,<1", "joblib>=1.3,<2",
    ],
    check=True,
)

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/Financial_Transaction_Anomaly_Detection")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Drive project folder: {PROJECT_DIR}")

## Embedded project functions

The next cell contains all analysis functions. The notebook runs without importing
Python files from the GitHub clone. The clone is a local project copy in the Colab
runtime; the dataset and generated results live in Google Drive.

In [ ]:
"""Core utilities for the financial transaction anomaly-detection project.

These functions are embedded so the notebook can run independently in Colab.
"""

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml, make_classification
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import RobustScaler


V_FEATURES = [f"V{i}" for i in range(1, 29)]
RAW_FEATURES = ["Time", *V_FEATURES, "Amount"]
REQUIRED_COLUMNS = [*RAW_FEATURES, "Class"]
MODEL_FEATURES = [*V_FEATURES, "LogAmount", "HourSin", "HourCos"]


@dataclass
class SplitData:
    train: pd.DataFrame
    validation: pd.DataFrame
    test: pd.DataFrame
    strategy: str


class PCAReconstructionDetector:
    """PCA detector whose anomaly score is scaled reconstruction error."""

    def __init__(self, n_components: float | int = 0.95, random_state: int = 42):
        self.n_components = n_components
        self.random_state = random_state
        self.pca_: PCA | None = None

    def fit(self, X: np.ndarray) -> "PCAReconstructionDetector":
        self.pca_ = PCA(
            n_components=self.n_components,
            svd_solver="full",
            random_state=self.random_state,
        )
        self.pca_.fit(X)
        return self

    def score_samples(self, X: np.ndarray) -> np.ndarray:
        if self.pca_ is None:
            raise RuntimeError("The detector must be fitted before scoring.")
        reconstructed = self.pca_.inverse_transform(self.pca_.transform(X))
        return np.mean(np.square(X - reconstructed), axis=1)


def validate_schema(df: pd.DataFrame, require_target: bool = True) -> None:
    required = REQUIRED_COLUMNS if require_target else RAW_FEATURES
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    if require_target:
        labels = set(pd.to_numeric(df["Class"], errors="coerce").dropna().unique())
        if not labels.issubset({0, 1}):
            raise ValueError("Class must contain only binary labels 0 and 1.")


def generate_synthetic_transactions(
    n_samples: int = 30_000,
    fraud_ratio: float = 0.006,
    random_state: int = 42,
) -> pd.DataFrame:
    """Generate a schema-compatible dataset for development smoke tests only."""
    X, y = make_classification(
        n_samples=n_samples,
        n_features=28,
        n_informative=12,
        n_redundant=8,
        n_repeated=0,
        n_clusters_per_class=3,
        weights=[1.0 - fraud_ratio, fraud_ratio],
        class_sep=1.45,
        flip_y=0.0005,
        random_state=random_state,
    )
    rng = np.random.default_rng(random_state)
    frame = pd.DataFrame(X, columns=V_FEATURES)
    frame["Time"] = np.sort(rng.uniform(0, 172_800, size=n_samples))
    normal_amount = rng.lognormal(mean=3.1, sigma=1.0, size=n_samples)
    fraud_multiplier = np.where(y == 1, rng.uniform(1.1, 3.0, n_samples), 1.0)
    frame["Amount"] = np.round(normal_amount * fraud_multiplier, 2)
    frame["Class"] = y.astype(int)
    return frame[["Time", *V_FEATURES, "Amount", "Class"]]


def load_transactions(
    csv_path: str | Path = "/content/creditcard.csv",
    *,
    allow_openml: bool = True,
    allow_synthetic: bool = False,
    synthetic_rows: int = 30_000,
    random_state: int = 42,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Load the ULB dataset locally, then OpenML, then an explicit smoke fallback."""
    candidates = [Path(csv_path), Path("/content/creditcard.csv")]
    for candidate in candidates:
        if candidate.exists():
            df = pd.read_csv(candidate)
            validate_schema(df)
            return df, {
                "source": str(candidate.resolve()),
                "is_synthetic": False,
                "note": "Loaded from local CSV.",
            }

    openml_error: str | None = None
    if allow_openml:
        try:
            bunch = fetch_openml(data_id=1597, as_frame=True, parser="auto")
            df = bunch.frame.copy()
            if "Class" not in df.columns and bunch.target is not None:
                df["Class"] = bunch.target
            df["Class"] = pd.to_numeric(df["Class"], errors="raise").astype(int)
            validate_schema(df)
            return df, {
                "source": "OpenML dataset 1597 (creditcard)",
                "is_synthetic": False,
                "note": "Downloaded through sklearn.fetch_openml.",
            }
        except Exception as exc:  # network/auth varies by execution platform
            openml_error = f"{exc.__class__.__name__}: {exc}"

    if allow_synthetic:
        df = generate_synthetic_transactions(
            n_samples=synthetic_rows,
            random_state=random_state,
        )
        return df, {
            "source": "Synthetic development data",
            "is_synthetic": True,
            "note": (
                "SMOKE TEST ONLY. Replace with the ULB creditcard.csv before "
                "submitting results."
            ),
            "openml_error": openml_error,
        }

    raise FileNotFoundError(
        "creditcard.csv was not found and OpenML could not be loaded. Download "
        "the ULB dataset to the Drive project folder or /content/creditcard.csv. "
        "OpenML error: "
        f"{openml_error or 'not attempted'}"
    )


def clean_transactions(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Coerce types, remove exact duplicates, and document data-quality actions."""
    validate_schema(df)
    clean = df[REQUIRED_COLUMNS].copy()
    for column in REQUIRED_COLUMNS:
        clean[column] = pd.to_numeric(clean[column], errors="coerce")
    missing_before = clean.isna().sum().to_dict()
    rows_before = len(clean)
    clean = clean.dropna().drop_duplicates().reset_index(drop=True)
    clean["Class"] = clean["Class"].astype(int)
    clean["Amount"] = clean["Amount"].clip(lower=0)
    validate_schema(clean)
    audit = {
        "rows_before": rows_before,
        "rows_after": len(clean),
        "rows_removed": rows_before - len(clean),
        "missing_before": {k: int(v) for k, v in missing_before.items()},
    }
    return clean, audit


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create robust amount and cyclical time features without using the label."""
    validate_schema(df, require_target=False)
    out = df.copy()
    seconds_in_day = np.mod(out["Time"].to_numpy(dtype=float), 86_400.0)
    hour = seconds_in_day / 3_600.0
    out["LogAmount"] = np.log1p(out["Amount"].clip(lower=0))
    out["HourSin"] = np.sin(2 * np.pi * hour / 24.0)
    out["HourCos"] = np.cos(2 * np.pi * hour / 24.0)
    return out


def chronological_split(
    df: pd.DataFrame,
    train_fraction: float = 0.70,
    validation_fraction: float = 0.15,
    random_state: int = 42,
) -> SplitData:
    """Use deployment-like time splits, with a documented stratified fallback."""
    from sklearn.model_selection import train_test_split

    ordered = df.sort_values("Time").reset_index(drop=True)
    train_end = int(len(ordered) * train_fraction)
    validation_end = int(len(ordered) * (train_fraction + validation_fraction))
    train = ordered.iloc[:train_end].copy()
    validation = ordered.iloc[train_end:validation_end].copy()
    test = ordered.iloc[validation_end:].copy()
    if all(part["Class"].nunique() == 2 for part in (train, validation, test)):
        return SplitData(train, validation, test, "chronological 70/15/15")

    train, remainder = train_test_split(
        ordered,
        test_size=1 - train_fraction,
        stratify=ordered["Class"],
        random_state=random_state,
    )
    relative_test = (1 - train_fraction - validation_fraction) / (1 - train_fraction)
    validation, test = train_test_split(
        remainder,
        test_size=relative_test,
        stratify=remainder["Class"],
        random_state=random_state,
    )
    return SplitData(
        train.sort_values("Time").copy(),
        validation.sort_values("Time").copy(),
        test.sort_values("Time").copy(),
        "stratified fallback because a chronological partition lacked a class",
    )


def cap_rows(
    X: np.ndarray,
    max_rows: int,
    random_state: int = 42,
) -> np.ndarray:
    if len(X) <= max_rows:
        return X
    rng = np.random.default_rng(random_state)
    return X[rng.choice(len(X), size=max_rows, replace=False)]


def fit_anomaly_models(
    X_train_normal_scaled: np.ndarray,
    *,
    fast_mode: bool = False,
    random_state: int = 42,
) -> dict[str, Any]:
    """Fit complementary global, local, and reconstruction-based detectors."""
    iso_rows = 25_000 if fast_mode else 100_000
    lof_rows = 10_000 if fast_mode else 45_000
    normal_iso = cap_rows(X_train_normal_scaled, iso_rows, random_state)
    normal_lof = cap_rows(X_train_normal_scaled, lof_rows, random_state)

    models: dict[str, Any] = {
        "Isolation Forest": IsolationForest(
            n_estimators=200 if fast_mode else 350,
            max_samples="auto",
            contamination="auto",
            random_state=random_state,
            n_jobs=-1,
        ).fit(normal_iso),
        "Local Outlier Factor": LocalOutlierFactor(
            n_neighbors=35,
            novelty=True,
            contamination="auto",
            n_jobs=-1,
        ).fit(normal_lof),
        "PCA Reconstruction": PCAReconstructionDetector(
            n_components=0.95,
            random_state=random_state,
        ).fit(normal_iso),
    }
    return models


def tune_anomaly_models(
    X_train_normal_scaled: np.ndarray,
    X_validation_scaled: np.ndarray,
    y_validation: np.ndarray,
    *,
    false_negative_cost: float = 25.0,
    false_positive_cost: float = 1.0,
    fast_mode: bool = False,
    random_state: int = 42,
) -> tuple[dict[str, Any], pd.DataFrame]:
    """Tune compact, explainable grids on validation cost within each family."""
    iso_rows = 25_000 if fast_mode else 100_000
    lof_rows = 10_000 if fast_mode else 35_000
    normal_iso = cap_rows(X_train_normal_scaled, iso_rows, random_state)
    normal_lof = cap_rows(X_train_normal_scaled, lof_rows, random_state)

    iso_grid = (
        [(120, "auto")]
        if fast_mode
        else [(200, "auto"), (350, "auto"), (350, 0.50), (350, 0.80)]
    )
    lof_grid = [20] if fast_mode else [20, 35, 50]
    pca_grid = [0.95] if fast_mode else [0.90, 0.95, 0.99]

    candidates: list[tuple[str, dict[str, Any], Any, np.ndarray]] = []
    for n_estimators, max_samples in iso_grid:
        params = {"n_estimators": n_estimators, "max_samples": max_samples}
        model = IsolationForest(
            n_estimators=n_estimators,
            max_samples=max_samples,
            contamination="auto",
            random_state=random_state,
            n_jobs=-1,
        ).fit(normal_iso)
        candidates.append(("Isolation Forest", params, model, X_validation_scaled))

    for n_neighbors in lof_grid:
        params = {"n_neighbors": n_neighbors}
        model = LocalOutlierFactor(
            n_neighbors=n_neighbors,
            novelty=True,
            contamination="auto",
            n_jobs=-1,
        ).fit(normal_lof)
        candidates.append(("Local Outlier Factor", params, model, X_validation_scaled))

    for n_components in pca_grid:
        params = {"variance_retained": n_components}
        model = PCAReconstructionDetector(
            n_components=n_components,
            random_state=random_state,
        ).fit(normal_iso)
        candidates.append(("PCA Reconstruction", params, model, X_validation_scaled))

    rows: list[dict[str, Any]] = []
    fitted: list[tuple[str, Any]] = []
    for family, params, model, X_val in candidates:
        scores = anomaly_score(family, model, X_val)
        choice = select_cost_threshold(
            y_validation,
            scores,
            false_negative_cost=false_negative_cost,
            false_positive_cost=false_positive_cost,
        )
        metrics = classification_metrics(y_validation, scores, choice["threshold"])
        rows.append(
            {
                "Family": family,
                "Parameters": json.dumps(params, sort_keys=True),
                "Validation_Cost": choice["validation_cost"],
                "F2": metrics["F2"],
                "PR_AUC": metrics["PR_AUC"],
                "Recall": metrics["Recall"],
                "Precision": metrics["Precision"],
                "Threshold": choice["threshold"],
            }
        )
        fitted.append((family, model))

    tuning_table = pd.DataFrame(rows)
    tuning_table["Selected"] = False
    selected: dict[str, Any] = {}
    for family in tuning_table["Family"].unique():
        family_rows = tuning_table.loc[tuning_table["Family"] == family]
        best_index = family_rows.sort_values(
            ["Validation_Cost", "F2", "PR_AUC"],
            ascending=[True, False, False],
        ).index[0]
        selected[family] = fitted[int(best_index)][1]
        tuning_table.loc[best_index, "Selected"] = True
    return selected, tuning_table.sort_values(
        ["Family", "Selected", "Validation_Cost"],
        ascending=[True, False, True],
    ).reset_index(drop=True)


def anomaly_score(model_name: str, model: Any, X_scaled: np.ndarray) -> np.ndarray:
    """Return scores where larger always means more anomalous."""
    raw = np.asarray(model.score_samples(X_scaled), dtype=float)
    if model_name == "PCA Reconstruction":
        return raw
    return -raw


def iqr_amount_score(df: pd.DataFrame, train_reference: pd.DataFrame) -> np.ndarray:
    """A transparent univariate statistical baseline using log transaction amount."""
    train_log = np.log1p(train_reference["Amount"].clip(lower=0))
    q1, q3 = train_log.quantile([0.25, 0.75])
    iqr = max(float(q3 - q1), 1e-9)
    center = float(train_log.median())
    return np.abs(np.log1p(df["Amount"].clip(lower=0)).to_numpy() - center) / iqr


def select_cost_threshold(
    y_true: Iterable[int],
    scores: Iterable[float],
    *,
    false_negative_cost: float = 25.0,
    false_positive_cost: float = 1.0,
) -> dict[str, float]:
    """Choose a validation-only threshold minimizing explicit business cost."""
    y = np.asarray(y_true, dtype=int)
    s = np.asarray(scores, dtype=float)
    precision, recall, thresholds = precision_recall_curve(y, s)
    precision = precision[:-1]
    recall = recall[:-1]
    positives = max(int(y.sum()), 1)
    tp = recall * positives
    predicted_positive = np.divide(
        tp,
        precision,
        out=np.full_like(tp, fill_value=float(len(y))),
        where=precision > 0,
    )
    fp = np.maximum(predicted_positive - tp, 0)
    fn = positives - tp
    costs = false_negative_cost * fn + false_positive_cost * fp
    best = int(np.nanargmin(costs))
    return {
        "threshold": float(thresholds[best]),
        "validation_cost": float(costs[best]),
        "false_negative_cost": float(false_negative_cost),
        "false_positive_cost": float(false_positive_cost),
    }


def classification_metrics(
    y_true: Iterable[int],
    scores: Iterable[float],
    threshold: float,
) -> dict[str, float | int]:
    y = np.asarray(y_true, dtype=int)
    s = np.asarray(scores, dtype=float)
    pred = (s >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "PR_AUC": float(average_precision_score(y, s)),
        "ROC_AUC": float(roc_auc_score(y, s)),
        "Precision": float(precision_score(y, pred, zero_division=0)),
        "Recall": float(recall_score(y, pred, zero_division=0)),
        "F1": float(f1_score(y, pred, zero_division=0)),
        "F2": float(fbeta_score(y, pred, beta=2, zero_division=0)),
        "False_Positive_Rate": float(fp / max(fp + tn, 1)),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Threshold": float(threshold),
    }


def precision_at_k(y_true: Iterable[int], scores: Iterable[float], k: int) -> float:
    y = np.asarray(y_true, dtype=int)
    s = np.asarray(scores, dtype=float)
    k = min(max(int(k), 1), len(y))
    indices = np.argpartition(s, -k)[-k:]
    return float(y[indices].mean())


def fit_supervised_benchmark(
    X_train: np.ndarray,
    y_train: np.ndarray,
    *,
    random_state: int = 42,
    fast_mode: bool = False,
) -> tuple[Any, str]:
    """Fit SMOTE+RF when available; otherwise use balanced class weights."""
    rf = RandomForestClassifier(
        n_estimators=180 if fast_mode else 400,
        max_depth=12,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=random_state,
        n_jobs=-1,
    )
    try:
        from imblearn.over_sampling import SMOTE
        from imblearn.pipeline import Pipeline

        model = Pipeline(
            steps=[
                (
                    "smote",
                    SMOTE(
                        sampling_strategy=0.05,
                        k_neighbors=5,
                        random_state=random_state,
                    ),
                ),
                ("random_forest", rf),
            ]
        )
        model.fit(X_train, y_train)
        return model, "SMOTE (training only) + class-weighted Random Forest"
    except ImportError:
        rf.fit(X_train, y_train)
        return rf, "Class-weighted Random Forest (imbalanced-learn unavailable)"


def save_detection_bundle(
    output_path: str | Path,
    *,
    model_name: str,
    model: Any,
    scaler: RobustScaler,
    threshold: float,
    metadata: dict[str, Any] | None = None,
) -> Path:
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    bundle = {
        "model_name": model_name,
        "model": model,
        "scaler": scaler,
        "threshold": float(threshold),
        "raw_features": RAW_FEATURES,
        "model_features": MODEL_FEATURES,
        "metadata": metadata or {},
    }
    joblib.dump(bundle, output)
    return output


def save_json(payload: dict[str, Any], output_path: str | Path) -> Path:
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return output

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import (
    PrecisionRecallDisplay,
    RocCurveDisplay,
    ConfusionMatrixDisplay,
)
from sklearn.preprocessing import RobustScaler

SEED = 42
DATA_PATH = PROJECT_DIR / "creditcard.csv"
ALLOW_SYNTHETIC_SMOKE_TEST = False  # Keep False for final results.
FAST_MODE = False                   # Set True only for pipeline checks.
FN_COST = 25.0                      # Relative cost of missed fraud.
FP_COST = 1.0                       # Relative cost of a manual review.

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)

## 1. Load and verify the data

Place `creditcard.csv` in the Drive project folder, or upload it to
`/content/creditcard.csv`. The loader checks those locations in that order, then
tries OpenML dataset 1597. Synthetic data is disabled by default and is only for
a fast smoke test.

In [ ]:
raw_df, source_info = load_transactions(
    DATA_PATH,
    allow_openml=True,
    allow_synthetic=ALLOW_SYNTHETIC_SMOKE_TEST,
    synthetic_rows=30_000,
    random_state=SEED,
)
display(source_info)
print(f"Raw shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns")

if source_info["is_synthetic"]:
    warnings.warn(
        "SYNTHETIC SMOKE-TEST DATA IS ACTIVE. Do not submit these outputs.",
        stacklevel=1,
    )

RUN_OUTPUT_DIR = PROJECT_DIR / (
    "synthetic_smoke" if source_info["is_synthetic"] else "real_data"
)
ARTIFACTS_DIR = RUN_OUTPUT_DIR / "artifacts"
REPORTS_DIR = RUN_OUTPUT_DIR / "reports"
FIGURES_DIR = RUN_OUTPUT_DIR / "figures"
for directory in (ARTIFACTS_DIR, REPORTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)
save_json(source_info, REPORTS_DIR / "source_info.json")
print(f"Results will be saved to: {RUN_OUTPUT_DIR}")

In [ ]:
# Data-quality profile before transformation.
profile = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing": raw_df.isna().sum(),
    "unique": raw_df.nunique(dropna=False),
})
display(profile)
print("Exact duplicate rows:", int(raw_df.duplicated().sum()))
display(raw_df.head())

## 2. Clean, explore, and engineer features

Exact duplicate records and rows with unusable numeric values are removed and reported.
The target is never imputed. Amount is log-transformed because transaction values are
strongly right-skewed. Time-of-day is encoded cyclically, avoiding the false assumption
that hour 23 is far from hour 0.

No fraud rows are deleted merely because they are anomalous: unusual observations are
the phenomenon being modelled.

In [ ]:
clean_df, cleaning_audit = clean_transactions(raw_df)
df = engineer_features(clean_df)
display(cleaning_audit)

class_summary = (
    df["Class"].value_counts().rename_axis("Class").to_frame("count")
)
class_summary["percentage"] = 100 * class_summary["count"] / len(df)
display(class_summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.countplot(data=df, x="Class", ax=axes[0], color="#3465a4")
axes[0].set_title("Severe class imbalance")
axes[0].set_yscale("log")

sns.histplot(data=df, x="Amount", hue="Class", bins=80, element="step",
             stat="density", common_norm=False, ax=axes[1])
axes[1].set_xlim(0, df["Amount"].quantile(0.99))
axes[1].set_title("Amount distribution (to 99th percentile)")

sns.histplot(data=df, x="LogAmount", hue="Class", bins=60, element="step",
             stat="density", common_norm=False, ax=axes[2])
axes[2].set_title("Log-transformed amount")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_class_and_amount.png", dpi=160, bbox_inches="tight")
plt.show()

## 3. Leakage-resistant train, validation, and test design

Transactions are sorted by `Time` and split 70%/15%/15%. This approximates production:
the model learns from earlier activity and scores later activity. If a small development
dataset places only one class in a partition, the helper uses a documented stratified
fallback.

- **Train:** fit scaler and models.
- **Validation:** select thresholds and compare models.
- **Test:** estimate final performance once after all choices are locked.

RobustScaler is fitted on the training partition only. Unsupervised detectors learn the
profile of training observations labelled normal; labels are not used inside their loss
functions. This is novelty detection and should not be mislabeled as fully label-free
unsupervised training.

In [ ]:
split = chronological_split(df, random_state=SEED)
print("Split strategy:", split.strategy)

split_profile = pd.DataFrame({
    name: {
        "rows": len(part),
        "frauds": int(part["Class"].sum()),
        "fraud_rate_pct": 100 * part["Class"].mean(),
        "start_time": float(part["Time"].min()),
        "end_time": float(part["Time"].max()),
    }
    for name, part in {
        "train": split.train,
        "validation": split.validation,
        "test": split.test,
    }.items()
}).T
display(split_profile)

scaler = RobustScaler()
X_train = scaler.fit_transform(split.train[MODEL_FEATURES])
X_validation = scaler.transform(split.validation[MODEL_FEATURES])
X_test = scaler.transform(split.test[MODEL_FEATURES])
y_train = split.train["Class"].to_numpy(dtype=int)
y_validation = split.validation["Class"].to_numpy(dtype=int)
y_test = split.test["Class"].to_numpy(dtype=int)
X_train_normal = X_train[y_train == 0]

## 4. Candidate methods

1. **IQR amount baseline:** interpretable but univariate and unable to identify fraud with
   an ordinary transaction value.
2. **Isolation Forest:** isolates rare observations through random feature splits; fast
   and suitable for high-dimensional tabular data.
3. **Local Outlier Factor (novelty mode):** identifies points with lower local density
   than their neighbours; it can capture local fraud patterns but is heavier at scale.
4. **PCA reconstruction error:** observations poorly represented by the normal-data
   subspace receive higher error scores.
5. **Supervised benchmark:** SMOTE is applied to the training partition only, followed by
   a class-weighted Random Forest. If `imbalanced-learn` is unavailable, class weights
   are used without SMOTE.

A compact hyperparameter grid is evaluated within each anomaly-model family. Isolation
Forest varies tree count and sample fraction, LOF varies neighbourhood size, and PCA
varies retained variance. The best configuration in each family minimizes validation
business cost, with F2 and PR-AUC as tie-breakers. This keeps tuning transparent and
computationally practical.

Accuracy is deliberately excluded from model selection because predicting every record
as normal would exceed 99% accuracy on this dataset while detecting no fraud.

In [ ]:
anomaly_models, tuning_table = tune_anomaly_models(
    X_train_normal,
    X_validation,
    y_validation,
    false_negative_cost=FN_COST,
    false_positive_cost=FP_COST,
    fast_mode=FAST_MODE,
    random_state=SEED,
)
display(tuning_table.round(5))

validation_scores = {
    name: anomaly_score(name, model, X_validation)
    for name, model in anomaly_models.items()
}
test_scores = {
    name: anomaly_score(name, model, X_test)
    for name, model in anomaly_models.items()
}

validation_scores["IQR Amount Baseline"] = iqr_amount_score(
    split.validation, split.train
)
test_scores["IQR Amount Baseline"] = iqr_amount_score(split.test, split.train)

In [ ]:
supervised_model, imbalance_strategy = fit_supervised_benchmark(
    split.train[MODEL_FEATURES].to_numpy(dtype=float),
    y_train,
    random_state=SEED,
    fast_mode=FAST_MODE,
)
print("Supervised imbalance strategy:", imbalance_strategy)
validation_scores["Supervised RF Benchmark"] = supervised_model.predict_proba(
    split.validation[MODEL_FEATURES].to_numpy(dtype=float)
)[:, 1]
test_scores["Supervised RF Benchmark"] = supervised_model.predict_proba(
    split.test[MODEL_FEATURES].to_numpy(dtype=float)
)[:, 1]

## 5. Validation-only threshold optimization

A prediction becomes an alert when its score is at or above the threshold. Thresholds
are selected by minimizing:

$$\text{Expected cost}=C_{FN}\times FN+C_{FP}\times FP$$

The illustrative cost ratio is 25:1 because a missed fraudulent transaction is assumed
to be materially more costly than reviewing a legitimate alert. A bank should replace
these values with observed fraud loss, recovery, investigation, and customer-friction
costs.

In [ ]:
threshold_choices = {}
validation_rows = []

for name, scores in validation_scores.items():
    choice = select_cost_threshold(
        y_validation,
        scores,
        false_negative_cost=FN_COST,
        false_positive_cost=FP_COST,
    )
    threshold_choices[name] = choice
    metrics = classification_metrics(y_validation, scores, choice["threshold"])
    metrics.update({
        "Model": name,
        "Validation_Cost": choice["validation_cost"],
        "Precision_at_100": precision_at_k(y_validation, scores, 100),
    })
    validation_rows.append(metrics)

validation_table = (
    pd.DataFrame(validation_rows)
    .set_index("Model")
    .sort_values(["F2", "PR_AUC"], ascending=False)
)
display(validation_table.round(5))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for name, scores in validation_scores.items():
    PrecisionRecallDisplay.from_predictions(
        y_validation, scores, name=name, ax=axes[0]
    )
    RocCurveDisplay.from_predictions(
        y_validation, scores, name=name, ax=axes[1]
    )
axes[0].set_title("Validation precision-recall curves")
axes[1].set_title("Validation ROC curves")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "validation_curves.png", dpi=160, bbox_inches="tight")
plt.show()

## 6. Locked test evaluation

The following cell applies each validation-selected threshold to the untouched test
partition. This avoids optimistic test-set threshold tuning. For operational selection,
PR-AUC and F2 are more informative than ROC-AUC under severe class imbalance; false
positives and review capacity remain essential controls.

In [ ]:
test_rows = []
for name, scores in test_scores.items():
    threshold = threshold_choices[name]["threshold"]
    metrics = classification_metrics(y_test, scores, threshold)
    metrics.update({
        "Model": name,
        "Precision_at_100": precision_at_k(y_test, scores, 100),
        "Estimated_Business_Cost": FN_COST * metrics["FN"] + FP_COST * metrics["FP"],
    })
    test_rows.append(metrics)

test_table = (
    pd.DataFrame(test_rows)
    .set_index("Model")
    .sort_values(["F2", "PR_AUC"], ascending=False)
)
display(test_table.round(5))
test_table.to_csv(REPORTS_DIR / "model_comparison_test.csv")

In [ ]:
anomaly_names = [*anomaly_models.keys(), "IQR Amount Baseline"]
best_anomaly_name = max(
    anomaly_names,
    key=lambda name: validation_table.loc[name, "F2"],
)
best_threshold = threshold_choices[best_anomaly_name]["threshold"]
best_predictions = (test_scores[best_anomaly_name] >= best_threshold).astype(int)

print("Best anomaly detector selected on validation F2:", best_anomaly_name)
fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_predictions,
    display_labels=["Normal", "Fraud"],
    cmap="Blues",
    values_format=",d",
    ax=ax,
)
ax.set_title(f"Locked test confusion matrix - {best_anomaly_name}")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "best_anomaly_confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

## 7. Operational interpretation

The best statistical model is not automatically the best control. A fraud operations
team must also consider alert volume, case-handling capacity, latency, stability, and
explainability. The next cell reports alert workload and fraud capture at the locked
threshold. Review capacity can also be expressed as a top-$k$ queue.

In [ ]:
best_metrics = test_table.loc[best_anomaly_name]
operating_summary = {
    "selected_anomaly_model": best_anomaly_name,
    "test_transactions": int(len(y_test)),
    "alerts": int(best_metrics["TP"] + best_metrics["FP"]),
    "frauds_captured": int(best_metrics["TP"]),
    "frauds_missed": int(best_metrics["FN"]),
    "legitimate_reviews": int(best_metrics["FP"]),
    "recall": float(best_metrics["Recall"]),
    "precision": float(best_metrics["Precision"]),
    "f2": float(best_metrics["F2"]),
    "pr_auc": float(best_metrics["PR_AUC"]),
    "relative_cost": float(best_metrics["Estimated_Business_Cost"]),
}
display(operating_summary)
save_json(operating_summary, REPORTS_DIR / "operating_summary.json")

## 8. Saved model and notebook scoring

The stronger of Isolation Forest and Local Outlier Factor is selected using
validation F2. The saved bundle contains its scaler, feature order, model, and
locked threshold. The following cell saves it to Google Drive, and the next
cell demonstrates scoring a transaction directly in this notebook. PCA remains
in the comparison, and IQR remains a transparent baseline.

In [ ]:
deployment_candidates = ["Isolation Forest", "Local Outlier Factor"]
deployable_name = max(
    deployment_candidates,
    key=lambda name: validation_table.loc[name, "F2"],
)
deployable_threshold = threshold_choices[deployable_name]["threshold"]
bundle_path = save_detection_bundle(
    ARTIFACTS_DIR / "fraud_detection_bundle.joblib",
    model_name=deployable_name,
    model=anomaly_models[deployable_name],
    scaler=scaler,
    threshold=deployable_threshold,
    metadata={
        "dataset_source": source_info["source"],
        "is_synthetic": source_info["is_synthetic"],
        "split_strategy": split.strategy,
        "imbalance_strategy": imbalance_strategy,
        "false_negative_cost": FN_COST,
        "false_positive_cost": FP_COST,
        "validation_metrics": validation_table.loc[deployable_name].to_dict(),
        "test_metrics": test_table.loc[deployable_name].to_dict(),
    },
)
print("Saved deployment bundle:", bundle_path)
print("Model bundle saved in Google Drive.")

In [ ]:
# Example payload and local score; values come from a test observation.
example_payload = split.test.iloc[0][["Time", *[f"V{i}" for i in range(1, 29)], "Amount"]]
example_frame = pd.DataFrame([example_payload.to_dict()])
example_features = engineer_features(example_frame)[MODEL_FEATURES]
example_score = anomaly_score(
    deployable_name,
    anomaly_models[deployable_name],
    scaler.transform(example_features),
)[0]
display({
    "anomaly_score": float(example_score),
    "threshold": float(deployable_threshold),
    "decision": "manual_review" if example_score >= deployable_threshold else "allow",
})

## 9. Monitoring, governance, and retraining plan

**Data controls**

- Enforce schema, numeric ranges, null-rate limits, and feature order.
- Track amount/time distributions and scoring-volume changes.
- Keep personally identifiable information outside analytics logs.

**Model controls**

- Monitor alert rate, score distribution, precision, recall, PR-AUC, fraud loss captured,
  false-positive customer friction, and case-review turnaround time.
- Calculate drift measures such as Population Stability Index when a stable reference
  window exists.
- Revalidate the threshold when fraud cost, investigation capacity, or fraud patterns
  change. Retraining must never use future labels relative to the evaluation window.

**Human oversight**

- The model prioritizes transactions for review; it should not autonomously deny a
  customer without policy controls and appeal mechanisms.
- Store model version, score, threshold, and reviewer outcome for auditability.
- Use champion/challenger testing and rollback before replacing the production model.

**Fairness limitation**

Protected demographic attributes are absent because features are anonymized. Therefore,
this benchmark cannot demonstrate demographic fairness. That absence is a limitation,
not evidence that the model is fair.

## 10. Conclusions

This project demonstrates that fraud detection is an operating-threshold problem as
much as a modelling problem. The analysis compares global, local, statistical, and
reconstruction-based detectors under a leakage-resistant evaluation design. It handles
class imbalance through fit-for-purpose metrics, cost-sensitive thresholding, normal-only
novelty training, and a SMOTE/class-weighted supervised reference.

The recommended model must be taken from the executed comparison table rather than
hard-coded in advance. A production rollout should begin as a decision-support control,
collect reviewer feedback, monitor drift, and recalibrate costs and thresholds before
any automated blocking action.

### Limitations and next steps

- The benchmark covers only two days and anonymizes most features.
- Fraud labels may arrive with delay in real operations.
- Customer, merchant, device, geography, and graph relationships are unavailable.
- Future work should test temporal cross-validation, calibrated supervised models,
  autoencoders, graph anomaly detection, explainability, and live drift monitoring.

## References

1. Machine Learning Group, Universite Libre de Bruxelles. *Credit Card Fraud Detection* dataset. Kaggle: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
2. AlmaBetter. *Mastering Advanced Machine Learning for Industry Applications* (course material): anomaly preprocessing, statistical/ML outlier detection, and fraud modelling.
3. AlmaBetter. *Mastering ML Infrastructure from Model Training to Scalable Deployment* (course material): model packaging, monitoring, and feedback loops.
4. Scikit-learn documentation for Isolation Forest, Local Outlier Factor, PCA, Random Forest, and model-evaluation metrics: https://scikit-learn.org/stable/
5. imbalanced-learn documentation for SMOTE and leakage-safe pipelines: https://imbalanced-learn.org/stable/